In [ ]:
import pandas as pd

def debug_prompt_text(input_file):
    # 데이터 로드
    df = pd.read_csv(input_file)
    # 중복 제거 (실제 파이프라인과 동일하게)
    unique_papers = df.drop_duplicates(subset=['논문ID']).head(3)
    
    print("="*30)
    print("🚀 [DEBUG] API 전송 예정 텍스트 샘플")
    print("="*30)
    
    for i, (_, row) in enumerate(unique_papers.iterrows()):
        # 실제 classify_batch 함수에서 만드는 것과 동일한 로직
        items_text = f"- 제목: {row['제목']} / 키워드: {row['키워드']}\n"
        
        print(f"[{i+1}번 논문 전송 텍스트]:")
        print(items_text)
        print("-" * 30)

# 실행 (상세데이터 파일명을 넣으세요)
debug_prompt_text('법학_AI_논문_상세정보_리스트.csv')

### LLM으로 분류

In [ ]:
import pandas as pd
import google.generativeai as genai
from pydantic import BaseModel
from typing import List
from tqdm import tqdm
from enum import Enum
import time
import os
import json
from dotenv import load_dotenv
from typing import List, Literal

import pandas as pd
from google import genai  # ⭐ 변경!
from google.genai import types  # ⭐ 추가
from pydantic import BaseModel
from typing import List, Literal
from tqdm import tqdm
import time
import os
import json
from dotenv import load_dotenv

# 0. 설정 로드
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")

print(f"현재 작업 경로: {os.getcwd()}")
print(f"✅ API 키 로드: {API_KEY[:4]}..." if API_KEY else "❌ API 키 없음")

# 1. Pydantic 모델 (동일)
class ClassificationItem(BaseModel):
    source_id: str
    category: Literal["형사법", "공법", "민사법", "기타"]

class BatchClassificationResponse(BaseModel):
    results: List[ClassificationItem]

# 2. 새 SDK로 API 설정
client = genai.Client(api_key=API_KEY)

# API 연결 테스트
try:
    print("🔌 API 연결 테스트 중...")
    test_response = client.models.generate_content(
        model='gemini-2.5-flash',  # 모델명 확인
        contents='테스트'
    )
    print(f"✅ API 연결 성공! 응답: {test_response.text[:30]}")
except Exception as e:
    print(f"❌ API 연결 실패: {e}")
    exit(1)

def classify_batch(batch_df):
    """20개의 논문을 하나의 프롬프트로 묶어 분류 요청"""
    items_text = ""
    for _, row in batch_df.iterrows():
        raw_keywords = str(row['키워드'])
        clean_keywords = raw_keywords.replace(']]>', '').strip()
        items_text += f"[ID: {row['source_id']}] 제목: {row['제목']} / 키워드: {clean_keywords}\n"
    
    print(f"📏 프롬프트 길이: {len(items_text)} 글자")
    
    prompt = f"""
    <역할>
    30년 이상의 경력을 지닌 대한민국의 법학가로서, 너는 법률 논문을 분류하는 작업을 진행해야 해.
    주어진 논문들의 제목과 키워드를 읽고 각 논문을 <분류 기준>을 따라 4개 카테고리 중 하나로 분류해줘.
    </역할>

    <분류 기준>
    1. 형사법: 형벌에 관한 사항을 규율하는 법 체계

    2. 공법: 국가, 지방자치단체 등 공적 기관 상호 간, 또는 국가·지방자치단체와 개인(사인) 사이의 관계를 규율하는 법 체계 중 형사법이 아닌 것 

    3. 민사법: 개인(사인) 간의 재산적·신분적 생활 관계를 규율하는 법 체계

    4. 기타: 위 세가지 분류 중 어느 하나에 명확하게 속하지 않거나 동시에 여러 분류에 속하는 법 체계
    </분류 기준>

    [대상 리스트]
    {items_text}
    
    각 논문에 대해 source_id와 category를 JSON 형식으로 반환해.
    """

    try:
        print(f"🔄 API 호출 시작... (배치 크기: {len(batch_df)})")
        start_time = time.time()
        
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=BatchClassificationResponse,
                temperature=0.0
            )
        )
        
        elapsed = time.time() - start_time
        print(f"✅ API 응답 완료 ({elapsed:.2f}초)")
        
        return response.text
        
    except Exception as e:
        print(f"❌ API 에러: {type(e).__name__} - {str(e)}")
        time.sleep(10)
        return None


def run_classification_pipeline(input_file, output_file):
    print(f"📋 {input_file} 로드 중...")
    df = pd.read_csv(input_file)
    
    if '논문ID' in df.columns:
        df = df.rename(columns={'논문ID': 'source_id'})
    
    unique_papers = df.drop_duplicates(subset=['source_id']).copy()

    print(f"✅ 총 {len(unique_papers)}건의 논문을 분류합니다.")
    print(f"📦 배치 크기: 20개씩, 총 {(len(unique_papers) + 19) // 20}개 배치")
    # 메인 실행 전에 추가
    print("🧪 첫 배치 테스트...")
    test_batch = unique_papers.head(5)  # 5개만 테스트
    test_result = classify_batch(test_batch)
    print(f"테스트 결과: {test_result[:200] if test_result else 'None'}")

    batch_size = 20
    success_count = 0
    fail_count = 0
    
    for i in tqdm(range(0, len(unique_papers), batch_size), desc="LLM 분류 진행 중"):
        batch_df = unique_papers.iloc[i : i + batch_size]
        print(f"\n--- 배치 {i//batch_size + 1} 시작 (논문 {i+1}~{min(i+batch_size, len(unique_papers))}) ---")
        
        json_response = classify_batch(batch_df)
        
        if json_response:
            try:
                parsed_data = json.loads(json_response)
                batch_results = pd.DataFrame(parsed_data['results'])
                
                batch_merged = batch_results.merge(
                    batch_df[['source_id', '제목', '발행연도', '키워드']],
                    on='source_id'
                )
                
                header = not os.path.exists(output_file)
                batch_merged.to_csv(output_file, index=False, mode='a', 
                                   header=header, encoding='utf-8-sig')
                
                success_count += len(batch_df)
                print(f"✅ 저장 완료 ({len(batch_df)}건)")
                
            except Exception as e:
                fail_count += len(batch_df)
                print(f"❌ 파싱 에러: {e}")
        else:
            fail_count += len(batch_df)
            print(f"⚠️ 이 배치는 건너뜁니다.")
        
        time.sleep(1)
    
    print(f"\n{'='*60}")
    print(f"🎯 처리 완료: 성공 {success_count}건 / 실패 {fail_count}건")
    print(f"{'='*60}")
    
    if os.path.exists(output_file):
        final_df = pd.read_csv(output_file)
        print(f"\n🏁 최종 결과 저장: {output_file}")
        print(final_df['category'].value_counts())

if __name__ == "__main__":

    run_classification_pipeline(
        input_file='법학_AI_논문_상세정보_리스트.csv', 
        output_file='AI_논문_분야별_분류_최종(예시x).csv'
    )

### BERT Topic으로 분류

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import os

def perform_kobert_topic_modeling(file_path):
    # 1. 데이터 로드 및 전처리
    df = pd.read_csv(file_path)
    
    # 텍스트 결합 (NaN 처리 및 공백 제거)
    df['combined_text'] = (
        df['제목'].fillna('') + " " + 
        df['초록'].fillna('') + " " + 
        df['키워드'].fillna('')
    ).str.strip()
    
    # 유효한 텍스트가 있는 행만 추출
    df = df[df['combined_text'] != ""].reset_index(drop=True)
    docs = df['combined_text'].tolist()

    # 2. Embedding 모델 로드 (jhgan/ko-sbert-sts)
    model_name = 'jhgan/ko-sbert-sts' 
    print(f"📡 모델 로딩 중: {model_name}...")
    
    embedding_model = SentenceTransformer(model_name)
    embedding_model.max_seq_length = 512 

    # 3. 벡터라이저 설정 (불용어 필터링)
    vectorizer_model = CountVectorizer(
        ngram_range=(1, 2), 
        stop_words=['인공지능', '분석', '연구', '고찰', '대한', '있으며', '따라']
    )

    # 4. BERTopic 모델 생성 (4개 클러스터)
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        nr_topics=4, 
        verbose=True
    )

    # 5. 모델 학습 및 결과 할당
    print("🚀 BERTopic 학습 시작...")
    topics, probs = topic_model.fit_transform(docs)
    df['Cluster_ID'] = topics

    # 6. 결과 저장 (combined_text 칼럼 제거)
    topic_info = topic_model.get_topic_info()
    
    # --- 수정된 부분: 저장용 데이터프레임에서 불필요한 칼럼 제거 ---
    df_output = df.drop(columns=['combined_text'])
    
    df_output.to_csv('법학_AI_논문_BERTopic_클러스터링.csv', index=False, encoding='utf-8-sig')
    topic_info.to_csv('클러스터별_대표키워드_정보.csv', index=False, encoding='utf-8-sig')
    
    print("-" * 50)
    print("✅ 분석 완료! 'combined_text'를 제외한 결과가 저장되었습니다.")
    return df_output, topic_info, topic_model

if __name__ == "__main__":
    file_name = '법학_AI_논문_상세정보_리스트.csv'
    if os.path.exists(file_name):
        result_df, info, model = perform_kobert_topic_modeling(file_name)
    else:
        print(f"❌ 파일을 찾을 수 없습니다: {file_name}")
        
# 실행 전 커널을 한 번 Restart 하시는 것을 권장합니다!
result_df, info, model = perform_kobert_topic_modeling('법학_AI_논문_상세정보_리스트.csv')

In [ ]:
# 1. Cluster_ID별 빈도수 계산
cluster_counts = result_df['Cluster_ID'].value_counts().sort_index()

# 2. 비율(%) 계산
cluster_percentages = (cluster_counts / len(result_df)) * 100

# 3. 데이터프레임으로 깔끔하게 정리
summary_df = pd.DataFrame({
    '개수(건)': cluster_counts,
    '비율(%)': cluster_percentages.round(2)
})

# 인덱스 이름 설정 (Cluster_ID)
summary_df.index.name = 'Cluster ID'

print("=== 클러스터별 분포 요약 ===")
print(summary_df)

# (선택 사항) 요약 결과도 CSV로 저장하고 싶다면:
# summary_df.to_csv('클러스터_비율_통계.csv', encoding='utf-8-sig')

In [ ]:
## 스펙트럴 클러스터링 (only법학만)

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.cluster import SpectralClustering
import os

def calculate_fixed_k_communities(filter_file, node_file, edge_file, k=4, output_file='KCI_AI_노드_커뮤니티_4개고정.csv'):
    print(f"0/3. 필터링 데이터 로드 중...")
    # 필터링 기준이 되는 논문 ID 리스트 로드
    df_filter = pd.read_csv(filter_file)
    valid_ids = set(df_filter['논문ID'].unique())
    print(f"   - 필터링 기준 논문 수: {len(valid_ids)}개")

    print(f"1/3. 데이터 로드 및 필터링 수행 중 (K={k})...")
    df_nodes = pd.read_csv(node_file)
    df_edges = pd.read_csv(edge_file)

    # 노드 필터링: node_id가 valid_ids에 속하는 경우만 유지
    df_nodes_filtered = df_nodes[df_nodes['node_id'].isin(valid_ids)].copy()
    filtered_node_ids = set(df_nodes_filtered['node_id'])
    
    # 엣지 필터링: source_id와 target_id_final 모두 valid_ids에 속하는 경우만 유지
    # (그래프의 무결성을 위해 양쪽 노드가 모두 존재해야 합니다)
    df_edges_filtered = df_edges[
        df_edges['source_id'].isin(filtered_node_ids) & 
        df_edges['target_id_final'].isin(filtered_node_ids)
    ].copy()

    print(f"   - 필터링 후 노드 수: {len(df_nodes_filtered)}개")
    print(f"   - 필터링 후 엣지 수: {len(df_edges_filtered)}개")

    # 그래프 생성 (무방향)
    G = nx.Graph()
    G.add_nodes_from(df_nodes_filtered['node_id'])
    edges = list(zip(df_edges_filtered['source_id'], df_edges_filtered['target_id_final']))
    G.add_edges_from(edges)

    # 노드 순서 보존을 위한 리스트
    node_list = list(G.nodes())
    
    # 2/3. 스펙트럴 클러스터링 실행
    if len(node_list) < k:
        print(f"⚠️ 경고: 필터링된 노드 수({len(node_list)})가 설정된 클러스터 수({k})보다 작습니다.")
        return None

    print(f"2/3. 스펙트럴 클러스터링 알고리즘 계산 중... (노드 {len(node_list)}개 대상)")
    adj_matrix = nx.to_numpy_array(G, nodelist=node_list)
    
    # n_clusters=k로 고정
    sc = SpectralClustering(
        n_clusters=k, 
        affinity='precomputed', 
        assign_labels='discretize', 
        random_state=42
    )
    labels = sc.fit_predict(adj_matrix)

    # 결과를 데이터프레임으로 변환
    df_partition = pd.DataFrame({'node_id': node_list, 'Community_ID': labels})

    print("3/3. 노드 메타데이터와 병합 및 저장 중...")
    # 필터링된 노드 정보와 병합 (원본 df_nodes가 아닌 df_nodes_filtered 사용)
    df_final = df_nodes_filtered.merge(df_partition, on='node_id', how='left')
    
    # 커뮤니티별 분포 확인
    print("\n📊 지정된 4개 커뮤니티별 노드 수:")
    print(df_final['Community_ID'].value_counts().sort_index())

    # 결과 저장
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n✅ 분석 완료! 결과 저장: {output_file}")

    return df_final

if __name__ == "__main__":
    # 파일 경로 설정
    filter_csv = '법학_AI_논문_상세정보_리스트.csv'
    node_csv = 'KCI_AI_전용_논문_노드.csv'
    edge_csv = 'KCI_AI_전용_인용_엣지.csv'
    
    if os.path.exists(filter_csv) and os.path.exists(node_csv) and os.path.exists(edge_csv):
        calculate_fixed_k_communities(filter_csv, node_csv, edge_csv, k=4)
    else:
        print("파일이 존재하지 않습니다. 경로를 확인해주세요.")

### 분류 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

def plot_yearly_trends_with_share(file_path):
    # 1. 데이터 로드 및 필터링
    df = pd.read_csv(file_path)
    df = df[df['발행연도'] >= 2016]
    
    # 2. 연도별/분류별 논문 '건수' 집계
    count_df = df.groupby(['발행연도', 'category']).size().unstack(fill_value=0)
    
    # 3. 점유율(%) 계산
    # 각 연도별 합계로 나누어 백분율을 구합니다.
    # 계산식: (해당 분야 건수 / 해당 연도 전체 건수) * 100
    share_df = count_df.div(count_df.sum(axis=1), axis=0) * 100
    
    # 4. CSV 파일 저장
    # '분야별_점유율_추이.csv'로 저장합니다.
    share_df.to_csv('분야별_점유율_추이.csv', encoding='utf-8-sig')
    print(f"✅ 점유율 계산 완료! '분야별_점유율_추이(예시x).csv' 파일이 생성되었습니다.")
    
    # 5. 시각화 (기존 로직 유지)
    plt.figure(figsize=(12, 7))
    count_df.plot(kind='area', stacked=True, alpha=0.7, ax=plt.gca())
    
    plt.title('AI 법학 논문의 분과별 발행 추이 (2016-2025)', fontsize=16, pad=20)
    plt.xlabel('발행 연도', fontsize=12)
    plt.ylabel('논문 수 (건)', fontsize=12)
    plt.legend(title='법학 분과', loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('분야별_발행추이.png', dpi=300)
    plt.show()


plot_yearly_trends_with_share('AI_논문_분야별_분류_최종(예시x).csv')